In [ ]:

Mahesh Karthik L
DS14418

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("Data set for DADS June.csv")

print("="*70)
print("SPENDDNA - TRANSACTION PARSER")
print("="*70)

print("Original Shape :", df.shape)
print("Duplicate Rows :", df.duplicated().sum())

before = len(df)

df = df.drop_duplicates()

duplicates_removed = before - len(df)

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹","",regex=False)
    .str.replace("Rs.","",regex=False)
    .str.replace(",","",regex=False)
    .str.strip()
)

df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)

df["Type"] = (
    df["Type"]
    .astype(str)
    .str.lower()
    .replace({
        "dr":"debit",
        "cr":"credit",
        "debit":"debit",
        "credit":"credit"
    })
)

df["Mode"] = df["Mode"].replace("",np.nan)

invalid_dates = df["Date"].isna().sum()
invalid_amounts = df["Amount"].isna().sum()

df = df.dropna(subset=["Date","Amount"])

df["Month"] = df["Date"].dt.strftime("%b")
df["Month_Number"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day_name()

df["Hour"] = (
    df["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

vendor_dict = {

"Swiggy":["SWIGGY","BUNDL"],

"Zomato":["ZOMATO"],

"Zepto":["ZEPTO"],

"Blinkit":["BLINKIT"],

"Instamart":["INSTAMART"],

"Amazon":["AMAZON","AMZN"],

"Flipkart":["FLIPKART"],

"Myntra":["MYNTRA"],

"Ajio":["AJIO"],

"Nykaa":["NYKAA"],

"BookMyShow":["BOOKMYSHOW"],

"Netflix":["NETFLIX"],

"Spotify":["SPOTIFY"],

"Prime Video":["PRIME"],

"Hotstar":["HOTSTAR","DISNEY"],

"YouTube":["YOUTUBE"],

"Uber":["UBER"],

"Ola":["OLA"],

"Rapido":["RAPIDO"],

"BMTC":["BMTC"],

"Indian Oil":["IOCL","INDIANOIL"],

"HP":["HPCL"],

"BPCL":["BPCL"],

"Reliance Fresh":["RELIANCE"],

"BigBasket":["BIGBASKET"],

"Dmart":["DMART"],

"Starbucks":["STARBUCKS"],

"Third Wave Coffee":["THIRDWAVE"],

"Cafe Coffee Day":["CCD"],

"Zerodha":["ZERODHA"],

"Groww":["GROWW"],

"Paytm":["PAYTM"],

"PhonePe":["PHONEPE"],

"Google Pay":["GPAY"],

"Electricity":["BESCOM"],

"Water Bill":["BWSSB"],

"Internet":["ACT"],

"Mobile Recharge":["JIO","AIRTEL","VI"],

"Rent":["RENT"],

"Cash Withdrawal":["ATM"],

"P2P Transfer":["OKAXIS","OKHDFC","OKICICI","OKSBI","OKYBL"]
}

def extract_vendor(description):

    text = str(description).upper()

    if "ATM" in text:
        return "Cash Withdrawal"

    for vendor, keywords in vendor_dict.items():

        for keyword in keywords:

            if keyword in text:
                return vendor

    if "UPI-" in text:
        return "P2P Transfer"

    return "Uncategorised"

df["Vendor"] = df["Description"].apply(extract_vendor)

print()
print("="*70)
print("TRANSACTION PARSER SUMMARY")
print("="*70)

print("Transactions Parsed :", len(df))
print("Duplicates Removed :", duplicates_removed)
print("Invalid Dates :", invalid_dates)
print("Invalid Amounts :", invalid_amounts)
print("Unique Vendors :", df["Vendor"].nunique())

print()
print("Top 10 Vendors")
print(df["Vendor"].value_counts().head(10))

print()
print(df.head())

SPENDDNA - TRANSACTION PARSER
Original Shape : (1328, 8)
Duplicate Rows : 18

TRANSACTION PARSER SUMMARY
Transactions Parsed : 143
Duplicates Removed : 18
Invalid Dates : 1167
Invalid Amounts : 0
Unique Vendors : 29

Top 10 Vendors
Vendor
Swiggy           28
Uncategorised    25
Zomato           16
Uber              7
Ola               6
Zepto             5
Starbucks         4
Flipkart          4
Nykaa             4
Zerodha           4
Name: count, dtype: int64

        Date   Time             Description   Type  Amount   Balance Mode  \
0 2024-01-01  03:11      AMAZON SELLER SVCS  debit  2462.0  678275.0  UPI   
3 2024-01-01  14:07    UPI-AMAN-8934@OKAXIS  debit  1828.0 -748745.0  UPI   
5 2024-01-01  14:48              BHIM ZEPTO  debit   625.0  677650.0  UPI   
6 2024-01-01  14:50  UPI-UBER-2426@HDFCBANK  debit   148.0  677020.0  UPI   
8 2024-02-01  05:18    POS SWIGGY BANGALORE  debit   537.0  676177.0  UPI   

         Ref Month  Month_Number       Day  Hour        Vendor  
0  TXN

In [4]:
category_map = {

"Swiggy":"Food Delivery",
"Zomato":"Food Delivery",

"Zepto":"Quick Commerce",
"Blinkit":"Quick Commerce",
"Instamart":"Quick Commerce",

"Amazon":"E-commerce",
"Flipkart":"E-commerce",
"Myntra":"E-commerce",
"Ajio":"E-commerce",
"Nykaa":"E-commerce",

"Uber":"Transport",
"Ola":"Transport",
"Rapido":"Transport",
"BMTC":"Transport",

"Starbucks":"Cafe",
"Third Wave Coffee":"Cafe",
"Cafe Coffee Day":"Cafe",

"BookMyShow":"Entertainment",

"Netflix":"Subscriptions",
"Spotify":"Subscriptions",
"Prime Video":"Subscriptions",
"Hotstar":"Subscriptions",
"YouTube":"Subscriptions",

"Electricity":"Utilities",
"Water Bill":"Utilities",
"Internet":"Utilities",
"Mobile Recharge":"Utilities",

"Reliance Fresh":"Groceries",
"BigBasket":"Groceries",
"Dmart":"Groceries",

"Zerodha":"Investments",
"Groww":"Investments",

"Indian Oil":"Fuel",
"HP":"Fuel",
"BPCL":"Fuel",

"Paytm":"Personal Transfer",
"PhonePe":"Personal Transfer",
"Google Pay":"Personal Transfer",
"P2P Transfer":"Personal Transfer",

"Cash Withdrawal":"Cash Withdrawal",

"Rent":"Rent"
}

df["Category"] = df["Vendor"].map(category_map)
df["Category"] = df["Category"].fillna("Others")

print("="*70)
print("CATEGORY SUMMARY")
print("="*70)

print(df["Category"].value_counts())

debit_df = df[df["Type"]=="debit"]
credit_df = df[df["Type"]=="credit"]

total_credit = credit_df["Amount"].sum()
total_debit = debit_df["Amount"].sum()

net_change = total_credit - total_debit

savings_rate = (net_change/total_credit)*100

transaction_count = len(df)

unique_vendors = df["Vendor"].nunique()

print()
print("="*70)
print("EXECUTIVE SUMMARY")
print("="*70)

print(f"Total Credits : Rs. {total_credit:,.2f}")
print(f"Total Debits  : Rs. {total_debit:,.2f}")
print(f"Net Change    : Rs. {net_change:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")
print(f"Transactions  : {transaction_count}")
print(f"Unique Vendors: {unique_vendors}")

category_spend = (
    debit_df
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

vendor_spend = (
    debit_df
    .groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

vendor_orders = (
    debit_df
    .groupby("Vendor")
    .size()
)

print()
print("="*70)
print("TOP 5 CATEGORIES")
print("="*70)

for category, amount in category_spend.head(5).items():

    percent = (amount/total_debit)*100

    print(f"{category:<20} Rs. {amount:>12,.2f}   {percent:>6.2f}%")

print()
print("="*70)
print("TOP 5 VENDORS")
print("="*70)

for vendor, amount in vendor_spend.head(5).items():

    orders = vendor_orders[vendor]

    print(f"{vendor:<20} Rs. {amount:>12,.2f}   ({orders} Orders)")

CATEGORY SUMMARY
Category
Food Delivery        44
Others               25
Transport            15
E-commerce           13
Quick Commerce       10
Cafe                  7
Personal Transfer     5
Investments           5
Groceries             4
Subscriptions         4
Utilities             3
Entertainment         3
Cash Withdrawal       3
Rent                  2
Name: count, dtype: int64

EXECUTIVE SUMMARY
Total Credits : Rs. 170,094.00
Total Debits  : Rs. 205,310.00
Net Change    : Rs. -35,216.00
Savings Rate  : -20.70%
Transactions  : 143
Unique Vendors: 29

TOP 5 CATEGORIES
Investments          Rs.    64,496.00    31.41%
E-commerce           Rs.    36,416.00    17.74%
Rent                 Rs.    36,000.00    17.53%
Food Delivery        Rs.    19,617.00     9.55%
Others               Rs.    17,254.00     8.40%

TOP 5 VENDORS
Zerodha              Rs.    60,000.00   (4 Orders)
Rent                 Rs.    36,000.00   (2 Orders)
Uncategorised        Rs.    17,254.00   (23 Orders)
Myntra    

In [5]:
month_order = ["Jan","Feb","Mar","Apr","May","Jun"]

monthly_trend = debit_df.pivot_table(
    values="Amount",
    index="Category",
    columns="Month",
    aggfunc="sum",
    fill_value=0
)

monthly_trend = monthly_trend.reindex(columns=month_order)

print("="*70)
print("MONTHLY SPENDING MATRIX")
print("="*70)

print(monthly_trend)

print()

growth = {}

for category in monthly_trend.index:

    first = monthly_trend.loc[category,"Jan"]
    last = monthly_trend.loc[category,"Jun"]

    if first == 0:
        growth[category] = 0
    else:
        growth[category] = ((last-first)/first)*100

largest_growth = max(growth,key=growth.get)
largest_decline = min(growth,key=growth.get)

print("="*70)
print("MONTHLY TREND ANALYSIS")
print("="*70)

print(f"Highest Growth Category : {largest_growth}")
print(f"Growth Percentage       : {growth[largest_growth]:.2f}%")

print()

print(f"Highest Decline Category: {largest_decline}")
print(f"Decline Percentage      : {growth[largest_decline]:.2f}%")

print()

hourly_matrix = debit_df.pivot_table(
    values="Amount",
    index="Category",
    columns="Hour",
    aggfunc="count",
    fill_value=0
)

print("="*70)
print("CATEGORY vs HOUR MATRIX")
print("="*70)

print(hourly_matrix)

print()

food_orders = debit_df[
    debit_df["Category"]=="Food Delivery"
]

food_hour = food_orders.groupby("Hour").size()

food_peak = food_hour.idxmax()

food_peak_orders = food_hour.max()

food_total_orders = len(food_orders)

food_peak_percent = (
    food_peak_orders /
    food_total_orders
) * 100

print("="*70)
print("TIME OF DAY PATTERNS")
print("="*70)

print(f"Food Delivery Peak Hour : {food_peak}:00")
print(f"Orders                  : {food_peak_orders}")
print(f"Percentage              : {food_peak_percent:.2f}%")

print()

cafe_orders = debit_df[
    debit_df["Category"]=="Cafe"
]

if len(cafe_orders)>0:

    cafe_hour = cafe_orders.groupby("Hour").size()

    cafe_peak = cafe_hour.idxmax()

    print(f"Cafe Peak Hour          : {cafe_peak}:00")

quick_orders = debit_df[
    debit_df["Category"]=="Quick Commerce"
]

if len(quick_orders)>0:

    quick_hour = quick_orders.groupby("Hour").size()

    quick_peak = quick_hour.idxmax()

    print(f"Quick Commerce Peak     : {quick_peak}:00")

print()

print("="*70)
print("FOOD DELIVERY MONTHLY TREND")
print("="*70)

food_month = (
    debit_df[
        debit_df["Category"]=="Food Delivery"
    ]
    .groupby("Month")["Amount"]
    .sum()
    .reindex(month_order)
)

max_amount = food_month.max()

for month, amount in food_month.items():

    if max_amount == 0:
        bars = ""
    else:
        bars = "*" * int((amount/max_amount)*20)

    print(f"{month:<5} Rs. {amount:>10,.2f}   {bars}")

MONTHLY SPENDING MATRIX
Month                 Jan     Feb      Mar     Apr      May     Jun
Category                                                           
Cafe                295.0   355.0      0.0   447.0    166.0     0.0
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0
E-commerce         6972.0   996.0   2147.0  3311.0      0.0  4868.0
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0
Groceries             0.0     0.0   1213.0     0.0      0.0     0.0
Investments           0.0     0.0      0.0     0.0      0.0     0.0
Others             2255.0   267.0   1416.0   513.0    753.0   794.0
Personal Transfer  1828.0     0.0    501.0     0.0      0.0     0.0
Quick Commerce     1616.0   697.0      0.0     0.0    502.0     0.0
Rent                  0.0     0.0  18000.0     0.0  18000.0     0.0
Subscriptions         0.0   354.0      0.0     0.0    431.0     0.0
Transport           826.

In [6]:
category_mean = debit_df.groupby("Category")["Amount"].transform("mean")

category_std = debit_df.groupby("Category")["Amount"].transform("std")

debit_df["Z_Score"] = (
    debit_df["Amount"] - category_mean
) / category_std

anomalies = debit_df[
    debit_df["Z_Score"] > 2
].sort_values(
    by="Z_Score",
    ascending=False
)

print("="*70)
print("TOP ANOMALIES")
print("="*70)

for _, row in anomalies.head(10).iterrows():

    print(
        f"{row['Date'].strftime('%d %b %Y')}   "
        f"{row['Vendor']:<20} "
        f"{row['Category']:<20} "
        f"Rs. {row['Amount']:>10,.2f}   "
        f"(z={row['Z_Score']:.2f})"
    )

food_total = category_spend.get("Food Delivery",0)
restaurant_total = category_spend.get("Restaurants",0)
cafe_total = category_spend.get("Cafe",0)

quick_total = category_spend.get("Quick Commerce",0)

ecommerce_total = category_spend.get("E-commerce",0)

investment_total = category_spend.get("Investments",0)

transport_total = category_spend.get("Transport",0)

food_percent = (
    (food_total + restaurant_total + cafe_total)
    / total_debit
) * 100

quick_percent = (
    quick_total / total_debit
) * 100

ecommerce_percent = (
    ecommerce_total / total_debit
) * 100

investment_percent = (
    investment_total / total_debit
) * 100

transport_percent = (
    transport_total / total_debit
) * 100

food_orders = debit_df[
    debit_df["Category"]=="Food Delivery"
]

late_orders = food_orders[
    (food_orders["Hour"]>=21) |
    (food_orders["Hour"]<=2)
]

if len(food_orders)>0:
    late_percent = (
        len(late_orders)
        / len(food_orders)
    ) * 100
else:
    late_percent = 0

subscription_vendors = debit_df[
    debit_df["Category"]=="Subscriptions"
]["Vendor"].nunique()

print()
print("="*70)
print("RAHUL'S SPENDING ARCHETYPES")
print("="*70)

if food_percent > 25:
    print(f"THE FOODIE ({food_percent:.1f}% on food)")

if quick_percent > 15:
    print(f"THE QUICK COMMERCE JUNKIE ({quick_percent:.1f}%)")

if ecommerce_percent > 15:
    print(f"THE SHOPAHOLIC ({ecommerce_percent:.1f}%)")

if investment_percent > 15:
    print(f"THE INVESTOR ({investment_percent:.1f}%)")

if late_percent > 50:
    print(f"THE LATE-NIGHT SNACKER ({late_percent:.1f}% food orders after 9 PM)")

if transport_percent > 10:
    print(f"THE CAB COMMUTER ({transport_percent:.1f}%)")

if subscription_vendors >= 5:
    print(f"THE SUBSCRIPTION LOVER ({subscription_vendors} active subscriptions)")

if savings_rate < 10:
    print(f"THE YOLO SPENDER ({savings_rate:.1f}% savings rate)")

if savings_rate > 40:
    print(f"THE DISCIPLINED SAVER ({savings_rate:.1f}% savings rate)")

TOP ANOMALIES
01 Sep 2024   Myntra               E-commerce           Rs.  10,745.00   (z=3.07)
03 Dec 2024   Uncategorised        Others               Rs.   2,430.00   (z=2.40)
01 Dec 2024   Uncategorised        Others               Rs.   2,329.00   (z=2.25)

RAHUL'S SPENDING ARCHETYPES
THE SHOPAHOLIC (17.7%)
THE INVESTOR (31.4%)
THE YOLO SPENDER (-20.7% savings rate)


/tmp/ipykernel_4232/613122929.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  debit_df["Z_Score"] = (


In [7]:
print()
print("="*70)
print("                     SPENDDNA REPORT")
print("                      RAHUL SHARMA")
print("="*70)

print(f"Period               : Jan 2024 - Jun 2024")
print(f"Transactions         : {transaction_count}")
print(f"Unique Vendors       : {unique_vendors}")

print()

print("="*70)
print("EXECUTIVE SUMMARY")
print("="*70)

print(f"Total Credits        : Rs. {total_credit:,.2f}")
print(f"Total Debits         : Rs. {total_debit:,.2f}")
print(f"Net Change           : Rs. {net_change:,.2f}")
print(f"Savings Rate         : {savings_rate:.2f}%")

if net_change < 0:
    print("Financial Status     : OVERSPENDING")
else:
    print("Financial Status     : SAVING")

print()

print("="*70)
print("TOP CATEGORIES")
print("="*70)

for category, amount in category_spend.head(5).items():

    percent = (amount / total_debit) * 100

    stars = "*" * int(percent)

    print(f"{category:<22} {stars:<25} {percent:>6.2f}%   Rs. {amount:,.2f}")

print()

print("="*70)
print("TOP VENDORS")
print("="*70)

for vendor, amount in vendor_spend.head(5).items():

    orders = vendor_orders[vendor]

    print(f"{vendor:<20} Rs. {amount:>12,.2f}   ({orders} Orders)")

print()

print("="*70)
print("TIME OF DAY PATTERNS")
print("="*70)

print(f"Food Delivery Peak Hour : {food_peak}:00")
print(f"Food Delivery Orders    : {food_peak_orders}")
print(f"Late Night Orders       : {late_percent:.2f}%")

if len(cafe_orders) > 0:
    print(f"Cafe Peak Hour          : {cafe_peak}:00")

if len(quick_orders) > 0:
    print(f"Quick Commerce Peak     : {quick_peak}:00")

print()

print("="*70)
print("FOOD DELIVERY MONTHLY TREND")
print("="*70)

for month, amount in food_month.items():

    if max_amount == 0:
        bar = ""
    else:
        bar = "*" * int((amount / max_amount) * 20)

    print(f"{month:<5} Rs. {amount:>10,.2f}   {bar}")

print()

print("="*70)
print("TOP ANOMALOUS TRANSACTIONS")
print("="*70)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{row['Date'].strftime('%d %b %Y')}   "
        f"{row['Vendor']:<20} "
        f"Rs. {row['Amount']:>10,.2f}   "
        f"Z-Score : {row['Z_Score']:.2f}"
    )

print()

print("="*70)
print("SPENDING ARCHETYPES")
print("="*70)

if food_percent > 25:
    print(f"THE FOODIE ({food_percent:.2f}% Food Spending)")

if quick_percent > 15:
    print(f"THE QUICK COMMERCE JUNKIE ({quick_percent:.2f}%)")

if ecommerce_percent > 15:
    print(f"THE SHOPAHOLIC ({ecommerce_percent:.2f}%)")

if investment_percent > 15:
    print(f"THE INVESTOR ({investment_percent:.2f}%)")

if late_percent > 50:
    print(f"THE LATE-NIGHT SNACKER ({late_percent:.2f}% Late Orders)")

if subscription_vendors >= 5:
    print(f"THE SUBSCRIPTION LOVER ({subscription_vendors} Active Subscriptions)")

if savings_rate < 10:
    print(f"THE YOLO SPENDER ({savings_rate:.2f}% Savings Rate)")

if savings_rate > 40:
    print(f"THE DISCIPLINED SAVER ({savings_rate:.2f}% Savings Rate)")

print()

print("="*70)
print("KEY INSIGHTS")
print("="*70)

monthly_loss = abs(net_change) / 6

print(f"1. Average monthly net spending : Rs. {monthly_loss:,.2f}")

top_category = category_spend.idxmax()

top_percent = (category_spend.max() / total_debit) * 100

print(f"2. Highest spending category is {top_category} ({top_percent:.2f}% of total debit).")

print(f"3. Total anomalies detected : {len(anomalies)}")

print()

print("="*70)
print("PROJECT COMPLETED SUCCESSFULLY")
print("="*70)
print("SpendDNA Analysis Finished.")


                     SPENDDNA REPORT
                      RAHUL SHARMA
Period               : Jan 2024 - Jun 2024
Transactions         : 143
Unique Vendors       : 29

EXECUTIVE SUMMARY
Total Credits        : Rs. 170,094.00
Total Debits         : Rs. 205,310.00
Net Change           : Rs. -35,216.00
Savings Rate         : -20.70%
Financial Status     : OVERSPENDING

TOP CATEGORIES
Investments            *******************************  31.41%   Rs. 64,496.00
E-commerce             *****************          17.74%   Rs. 36,416.00
Rent                   *****************          17.53%   Rs. 36,000.00
Food Delivery          *********                   9.55%   Rs. 19,617.00
Others                 ********                    8.40%   Rs. 17,254.00

TOP VENDORS
Zerodha              Rs.    60,000.00   (4 Orders)
Rent                 Rs.    36,000.00   (2 Orders)
Uncategorised        Rs.    17,254.00   (23 Orders)
Myntra               Rs.    13,629.00   (2 Orders)
Swiggy               Rs.  